# BushidoMythos — Finance Pretraining on Colab

**Enable GPU before running this notebook:**  
`Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 or A100)`

| GPU | dtype | batch x seq | Approx. speed |
|-----|-------|-------------|---------------|
| T4  (16 GB, SM 7.5) | float16 | 16 x 256 | ~150 steps/min |
| L4 / A10G (24 GB, SM 8.6) | bfloat16 | 32 x 512 | ~400 steps/min |
| A100 (40 GB, SM 8.0) | bfloat16 | 32 x 1024 | ~300 steps/min |
| A100 (80 GB, SM 8.0) | bfloat16 | 32 x 1024 | ~300 steps/min |

Mac (MPS) speed: ~30 steps/min (batch=4) — **about 5-25x faster on Colab GPUs**


## 1. Setup


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Set the repository path on Google Drive.
REPO = "/content/drive/Othercomputers/My Mac/bushido-mythos"

os.chdir(REPO)
print(f"Working directory: {os.getcwd()}")
!ls


Working directory: /content/drive/Othercomputers/My Mac/bushido-mythos
backup.sh		   data        loss_curve.png	 training
bushido_mythos		   docs        __pycache__	 venv
bushido_mythos_banner.svg  example.py  pyproject.toml	 venv_312
chat.py			   examples    README.md
checkpoints		   GEMINI.md   requirements.txt
colab_finance_train.ipynb  LICENSE     tests


In [3]:
!pip install -q transformers datasets
!pip install -q bitsandbytes  # optional: --optim8bit (8-bit AdamW, CUDA)
print("Done.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00:00:0100:01
Done.


In [4]:
import torch, os

if not torch.cuda.is_available():
    raise RuntimeError("No GPU was detected. Change the runtime type and enable GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
cc_major = torch.cuda.get_device_properties(0).major   # compute capability
is_ampere_plus = cc_major >= 8  # A100/A10G/H100 support native bfloat16

print(f"GPU            : {gpu_name}")
print(f"VRAM           : {vram_gb:.1f} GB")
print(f"Compute Cap.   : SM {cc_major}.x")
print(f"BF16 (native)  : {'Yes (Ampere+)' if is_ampere_plus else 'No -> using float16 (T4/V100)'}")

# Safe settings by GPU architecture.
# With seq_len=1024 + 8 loops, attention matrices are large, so keep batch conservative.
if vram_gb >= 60:                        # A100 80GB / H100 80GB
    BATCH_SIZE = 32
    SEQ_LEN    = 1024
    USE_COMPILE = True
elif vram_gb >= 38:                      # A100 40GB / H100 40GB
    BATCH_SIZE = 32
    SEQ_LEN    = 1024
    USE_COMPILE = True
elif is_ampere_plus and vram_gb >= 20:   # A10G / L4 24GB
    BATCH_SIZE = 16
    SEQ_LEN    = 512
    USE_COMPILE = True
elif is_ampere_plus:                     # A100 16GB-class devices
    BATCH_SIZE = 8
    SEQ_LEN    = 512
    USE_COMPILE = True
else:                                    # T4 / V100 (Turing/Volta, SM < 8)
    BATCH_SIZE = 16
    SEQ_LEN    = 256
    USE_COMPILE = False  # T4 does not support bfloat16 compile reliably; skip compile.

# Effective batch size = BATCH_SIZE x GRAD_ACCUM_STEPS.
# This increases the effective batch without increasing activation memory.
if vram_gb >= 38:                        # A100 40/80GB, H100
    GRAD_ACCUM_STEPS = 4                 # eff_batch = 32 x 4 = 128
elif is_ampere_plus and vram_gb >= 20:   # A10G / L4 24GB
    GRAD_ACCUM_STEPS = 8                 # eff_batch = 16 x 8 = 128
elif is_ampere_plus:                     # A100 16GB-class devices
    GRAD_ACCUM_STEPS = 16                # eff_batch = 8 x 16 = 128
else:                                    # T4 / V100
    GRAD_ACCUM_STEPS = 8                 # eff_batch = 16 x 8 = 128

# Reduce CUDA memory fragmentation.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

eff = BATCH_SIZE * GRAD_ACCUM_STEPS
print(f"\nSettings: --batch_size {BATCH_SIZE}  --grad_accum_steps {GRAD_ACCUM_STEPS}  eff_batch={eff}  --seq_len {SEQ_LEN}  --compile={USE_COMPILE}")

# Whether to include Dolly (databricks-dolly-15k, CC BY-SA 3.0).
# For commercial or production use, check attribution and share-alike obligations before enabling.
INCLUDE_DOLLY = True

# Enable gradient checkpointing to reduce VRAM usage (~7x for 8 loops, at cost of ~35% slower training).
# Set to True if you get OOM errors, or when training with more loop iterations.
USE_GRAD_CHECKPOINT = True

# Loop curriculum (experimental). off = model default;
# "fixed" = pin n_loops to max_loop_iters (clean baseline);
# "curriculum" = phase-based variable recurrence (faster + depth extrapolation). See README.
LOOP_SCHEDULE = "curriculum"
LOOP_TAIL_MAX = 12  # curriculum: max loops in the Phase 2+ upward tail
LOOP_TAIL_P = 0.2   # curriculum: probability of sampling the tail
LOOP_SEED = 0       # curriculum: sampler seed (deterministic per step)
# Base model depth must cover the curriculum tail so loops 9-12 get their own depth-LoRA.
# When LOOP_SCHEDULE='curriculum', the base is built with max_loop_iters = LOOP_TAIL_MAX.
MAX_LOOP_ITERS = LOOP_TAIL_MAX if LOOP_SCHEDULE == "curriculum" else 8

# 8-bit optimizer (CUDA + bitsandbytes): optimizer states 8->2 byte/param, near-lossless
OPTIM8BIT = True
# Memory replay (anti-forgetting): fraction of Phase 2+ batches from a general-language anchor
# 例: 0.05 = 5%。破滅的忘却(汎用性能の劣化)を抑える。0.0=無効
REPLAY_RATIO = 0.05

# ACT カリキュラム(動的 act_threshold / ponder cost): 浅→深で深い推論を解禁。
# anchor=0 + warmup_frac=0.73 で phase1-2 でランプし phase3-5(finance)を深く回す。
# フェーズを別セル(別プロセス)で回すため、anchor は全セル共通の固定値(0)を渡す。
ACT_CURRICULUM = True
ACT_THRESHOLD_START = 0.5
ACT_WARMUP_FRAC = 0.73
ACT_ANCHOR_STEP = 0
PONDER_WEIGHT_START = 0.03
PONDER_WEIGHT_END = 0.0

# ACT カリキュラムと torch.compile は両立しない(閾値変更ごとに再コンパイル)。
# 当面は ACT_CURRICULUM=True のとき compile を無効化する。
if ACT_CURRICULUM:
    USE_COMPILE = False

# 対照実験(A/B): カリキュラム無しランは別ディレクトリに保存し、
# curriculum 版の結果(finance_a100_v2)を上書きしないようにする。
CKPT_SUBDIR = "finance_a100_v2" if ACT_CURRICULUM else "finance_a100_v2_noact"

if INCLUDE_DOLLY:
    print('Dolly (CC BY-SA 3.0): enabled — verify the license obligations.')
else:
    print('Dolly: disabled (no --include_dolly)')


GPU            : Tesla T4
VRAM           : 15.6 GB
Compute Cap.   : SM 7.x
BF16 (native)  : No -> using float16 (T4/V100)

Settings: --batch_size 16  --grad_accum_steps 8  eff_batch=128  --seq_len 256  --compile=False
Dolly (CC BY-SA 3.0): enabled — verify the license obligations.


## 1a. Cache Setup (NVMe Speedup)

Tokenized dataset cache is copied from Drive to `/content/cache` (NVMe) at startup.
This reduces per-restart overhead: Drive FUSE ~10 MB/s vs NVMe ~3 GB/s.
**Run once per session before any training cell.**

In [5]:
import shutil
from pathlib import Path

LOCAL_CACHE = "/content/cache"
DRIVE_CACHE = f"{REPO}/.cache"

Path(LOCAL_CACHE).mkdir(parents=True, exist_ok=True)

drive_path = Path(DRIVE_CACHE)
if drive_path.exists():
    files = [f for f in drive_path.rglob("*") if f.is_file()]
    if files:
        size_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f"Copying {len(files)} cache files ({size_mb:.0f} MB) from Drive \u2192 /content/cache ...")
        shutil.copytree(DRIVE_CACHE, LOCAL_CACHE, dirs_exist_ok=True)
        print("Done. Training will use fast NVMe cache.")
    else:
        print("Drive cache directory exists but is empty. Fresh cache will be built at /content/cache.")
else:
    print("No Drive cache found. Fresh cache will be built at /content/cache on first training run.")

print(f"Cache directory: {LOCAL_CACHE}")


Copying 8 cache files (7053 MB) from Drive → /content/cache ...
Done. Training will use fast NVMe cache.
Cache directory: /content/cache


## 2. Create the Starting Checkpoint (A100-Optimized Model)

This creates the starting checkpoint for a larger model: `dim=768 / n_heads=12 / n_experts=28 / expert_dim=768 / ~99M params`.  
`loop_curriculum=True` (random-depth training) is also enabled.  
Output: `checkpoints/a100_v2_gpt2vocab/final.pt` (about 395 MB).  
**Run this once before Phase 1.**


> **Depth note:** with `LOOP_SCHEDULE="curriculum"`, the base is built at `max_loop_iters = LOOP_TAIL_MAX` (e.g. 12) so the tail (9–12) gets its own depth-LoRA. The Phase cells' auto-create only runs **if the base file is missing** — if a `max_loop_iters=8` base already exists at `BASE_CKPT`, delete it (or use a fresh `BASE_CKPT`/`CKPT_DIR` path) before running the curriculum experiment, otherwise the stale 8-loop base is reused.

In [6]:
!python training/make_base_ckpt.py --max_loop_iters {MAX_LOOP_ITERS}

モデル構成:
  seed=42
  attn_type=mla  dim=768  n_heads=12  n_kv_heads=4
  max_seq_len=1024  max_loop_iters=12
  n_experts=28  expert_dim=768
  loop_curriculum=True
  act_aux_loss_weight=0.001  (ACT warm-start; scale up to 0.01 after Phase 1)
  パラメータ数: 98.6M
GPT-2 small の埋め込みをロード中...
  ローカルキャッシュなし → ネットワークからダウンロードします
config.json: 100% 665/665 [00:00<00:00, 3.02MB/s]
model.safetensors: 100% 548M/548M [00:03<00:00, 171MB/s] 
Loading weights: 100% 148/148 [00:00<00:00, 9673.63it/s]
  GPT-2 埋め込み初期化完了 (shape: torch.Size([50257, 768]))

保存完了: checkpoints/a100_v2_gpt2vocab/final.pt  (394.8 MB)


## 3. Run All 5 Phases at Once (Phase 1-5)


In [ ]:
import subprocess, sys
from pathlib import Path

CKPT_DIR  = f"{REPO}/checkpoints/{CKPT_SUBDIR}"
BASE_CKPT = f"{REPO}/checkpoints/a100_v2_gpt2vocab/final.pt"

# Auto-create base checkpoint if missing
if not Path(BASE_CKPT).exists():
    print(f"Base checkpoint not found. Running make_base_ckpt.py ...")
    r = subprocess.run(
        [sys.executable, "training/make_base_ckpt.py", "--out", BASE_CKPT, "--max_loop_iters", str(MAX_LOOP_ITERS)],
        check=True,
    )
    print(f"Created: {BASE_CKPT}")

cmd = [
    sys.executable, "training/finance_pretrain.py",
    "--base_ckpt",    BASE_CKPT,
    "--ckpt_dir",     CKPT_DIR,
    "--phase",        "0",
    "--phase1_steps", "30000",
    "--phase2_steps", "8000",
    "--phase2_openwebmath_rows", "80000",
    "--phase2_orca_ratio",       "35",
    "--phase3_steps", "8000",
    "--phase4_steps", "3000",
    "--phase5_steps", "3000",
    "--batch_size",   str(BATCH_SIZE),
    "--grad_accum_steps", str(GRAD_ACCUM_STEPS),
    "--seq_len",      str(SEQ_LEN),
    "--dtype",        "auto",
    "--log_every",    "100",
    "--mem_log_every", "100",
    "--cache_dir",    "/content/cache",
    "--save_every",   "2000",
    "--log_file",     f"{CKPT_DIR}/train.log",
    "--auto_resume",
]
if INCLUDE_DOLLY:
    cmd.append("--include_dolly")
if USE_COMPILE:
    cmd.append("--compile")
if USE_GRAD_CHECKPOINT:
    cmd.append("--grad_checkpoint")
if LOOP_SCHEDULE != "off":
    cmd += ["--loop_schedule", LOOP_SCHEDULE]
if LOOP_SCHEDULE == "curriculum":
    cmd += ["--loop_tail_max", str(LOOP_TAIL_MAX), "--loop_tail_p", str(LOOP_TAIL_P), "--loop_seed", str(LOOP_SEED)]
if OPTIM8BIT:
    cmd.append("--optim8bit")
if REPLAY_RATIO > 0:
    cmd += ["--replay_ratio", str(REPLAY_RATIO)]

if ACT_CURRICULUM:
    cmd += [
        "--act_curriculum",
        "--act_anchor_step", str(ACT_ANCHOR_STEP),
        "--act_threshold_start", str(ACT_THRESHOLD_START),
        "--act_warmup_frac", str(ACT_WARMUP_FRAC),
        "--ponder_weight_start", str(PONDER_WEIGHT_START),
        "--ponder_weight_end", str(PONDER_WEIGHT_END),
    ]

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


Command: /usr/bin/python3 training/finance_pretrain.py --base_ckpt /content/drive/Othercomputers/My Mac/bushido-mythos/checkpoints/a100_v2_gpt2vocab/final.pt --ckpt_dir /content/drive/Othercomputers/My Mac/bushido-mythos/checkpoints/finance_a100_v2 --phase 0 --phase1_steps 30000 --phase2_steps 8000 --phase2_openwebmath_rows 80000 --phase2_orca_ratio 35 --phase3_steps 8000 --phase4_steps 3000 --phase5_steps 3000 --batch_size 16 --grad_accum_steps 8 --seq_len 256 --dtype auto --log_every 100 --mem_log_every 100 --cache_dir /content/cache --save_every 2000 --log_file /content/drive/Othercomputers/My Mac/bushido-mythos/checkpoints/finance_a100_v2/train.log --auto_resume --include_dolly --grad_checkpoint --loop_schedule curriculum --loop_tail_max 12 --loop_tail_p 0.2 --loop_seed 0 --optim8bit --replay_ratio 0.05 --act_curriculum --act_anchor_step 0 --act_threshold_start 0.5 --act_warmup_frac 0.73 --ponder_weight_start 0.03 --ponder_weight_end 0.0
--------------------------------------------

## 4. Phase 1 — WikiText-103 (General Language)

Training starts from `checkpoints/a100_v2_gpt2vocab/final.pt` at step 0.  
After interruption, `--auto_resume` can resume from the latest checkpoint.


In [ ]:
import subprocess, sys
from pathlib import Path

CKPT_DIR  = f"{REPO}/checkpoints/{CKPT_SUBDIR}"
BASE_CKPT = f"{REPO}/checkpoints/a100_v2_gpt2vocab/final.pt"

# Auto-create base checkpoint if missing
if not Path(BASE_CKPT).exists():
    print(f"Base checkpoint not found. Running make_base_ckpt.py ...")
    subprocess.run(
        [sys.executable, "training/make_base_ckpt.py", "--out", BASE_CKPT, "--max_loop_iters", str(MAX_LOOP_ITERS)],
        check=True,
    )
    print(f"Created: {BASE_CKPT}")

cmd = [
    sys.executable, "training/finance_pretrain.py",
    "--base_ckpt",    BASE_CKPT,
    "--ckpt_dir",     CKPT_DIR,
    "--phase",        "1",
    "--phase1_steps", "30000",
    "--phase2_steps", "8000",
    "--batch_size",   str(BATCH_SIZE),
    "--grad_accum_steps", str(GRAD_ACCUM_STEPS),
    "--seq_len",      str(SEQ_LEN),
    "--dtype",        "auto",
    "--log_every",    "100",
    "--mem_log_every", "100",
    "--cache_dir",    "/content/cache",
    "--save_every",   "2000",
    "--log_file",     f"{CKPT_DIR}/train.log",
    "--auto_resume",
]
if USE_COMPILE:
    cmd.append("--compile")
if USE_GRAD_CHECKPOINT:
    cmd.append("--grad_checkpoint")
if LOOP_SCHEDULE != "off":
    cmd += ["--loop_schedule", LOOP_SCHEDULE]
if LOOP_SCHEDULE == "curriculum":
    cmd += ["--loop_tail_max", str(LOOP_TAIL_MAX), "--loop_tail_p", str(LOOP_TAIL_P), "--loop_seed", str(LOOP_SEED)]
if OPTIM8BIT:
    cmd.append("--optim8bit")
if REPLAY_RATIO > 0:
    cmd += ["--replay_ratio", str(REPLAY_RATIO)]

if ACT_CURRICULUM:
    cmd += [
        "--act_curriculum",
        "--act_anchor_step", str(ACT_ANCHOR_STEP),
        "--act_threshold_start", str(ACT_THRESHOLD_START),
        "--act_warmup_frac", str(ACT_WARMUP_FRAC),
        "--ponder_weight_start", str(PONDER_WEIGHT_START),
        "--ponder_weight_end", str(PONDER_WEIGHT_END),
    ]

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


## 5. Phase 2 — Reasoning Mix

Run this after Phase 1 completes. Training continues from `phase1_final.pt`.

- **OpenWebMath** (`open-web-math/open-web-math`) — mathematical, proof-style, and quantitative reasoning text
- **Orca Math** (`microsoft/orca-math-word-problems-200k`) — math word problems with CoT-style solutions
- **Dolly** (`databricks/databricks-dolly-15k`, CC BY-SA 3.0) — enabled only when `INCLUDE_DOLLY=True`


In [ ]:
import subprocess, sys
from pathlib import Path

CKPT_DIR  = f"{REPO}/checkpoints/{CKPT_SUBDIR}"
BASE_CKPT = f"{REPO}/checkpoints/a100_v2_gpt2vocab/final.pt"
RESUME    = f"{CKPT_DIR}/phase1_final.pt"

if not Path(RESUME).exists():
    raise FileNotFoundError(
        f"Phase 1 checkpoint not found: {RESUME}\n"
        "Run the Phase 1 cell first and wait for it to complete."
    )

cmd = [
    sys.executable, "training/finance_pretrain.py",
    "--base_ckpt",    BASE_CKPT,
    "--ckpt_dir",     CKPT_DIR,
    "--phase",        "2",
    "--phase1_steps", "30000",
    "--phase2_steps", "8000",
    "--phase2_openwebmath_rows", "80000",
    "--phase2_orca_ratio",       "35",
    "--batch_size",   str(BATCH_SIZE),
    "--grad_accum_steps", str(GRAD_ACCUM_STEPS),
    "--seq_len",      str(SEQ_LEN),
    "--dtype",        "auto",
    "--log_every",    "100",
    "--mem_log_every", "100",
    "--cache_dir",    "/content/cache",
    "--save_every",   "2000",
    "--log_file",     f"{CKPT_DIR}/train.log",
    "--resume",       RESUME,
]
if INCLUDE_DOLLY:
    cmd.append("--include_dolly")
if USE_COMPILE:
    cmd.append("--compile")
if USE_GRAD_CHECKPOINT:
    cmd.append("--grad_checkpoint")
if LOOP_SCHEDULE != "off":
    cmd += ["--loop_schedule", LOOP_SCHEDULE]
if LOOP_SCHEDULE == "curriculum":
    cmd += ["--loop_tail_max", str(LOOP_TAIL_MAX), "--loop_tail_p", str(LOOP_TAIL_P), "--loop_seed", str(LOOP_SEED)]
if OPTIM8BIT:
    cmd.append("--optim8bit")
if REPLAY_RATIO > 0:
    cmd += ["--replay_ratio", str(REPLAY_RATIO)]

if ACT_CURRICULUM:
    cmd += [
        "--act_curriculum",
        "--act_anchor_step", str(ACT_ANCHOR_STEP),
        "--act_threshold_start", str(ACT_THRESHOLD_START),
        "--act_warmup_frac", str(ACT_WARMUP_FRAC),
        "--ponder_weight_start", str(PONDER_WEIGHT_START),
        "--ponder_weight_end", str(PONDER_WEIGHT_END),
    ]

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


## 6. Phase 3, 4 & 5 — Finance Domain Mix + SFT

Run this after Phase 2 completes. Training continues from `phase2_final.pt`.

- **Phase 3** — `ashraq/financial-news-articles` (financial news) + `gbharti/finance-alpaca` (21K finance instruction examples) as a plain-text mix
- **Phase 4** — `FinGPT/fingpt-forecaster-dow30-202305-202405` + `FinGPT/fingpt-sentiment-train` (trading-method SFT / ~78K pairs / 3,000 steps)
- **Phase 5** — `FinGPT/fingpt-fiqa_qa` (finance QA / final risk-management tuning / 3,000 steps)

The prompt format is fixed:
```
### Instruction:
Explain risk management in trading.

### Response:
```


In [ ]:
import subprocess, sys
from pathlib import Path

CKPT_DIR  = f"{REPO}/checkpoints/{CKPT_SUBDIR}"
BASE_CKPT = f"{REPO}/checkpoints/a100_v2_gpt2vocab/final.pt"
RESUME    = f"{CKPT_DIR}/phase2_final.pt"

if not Path(RESUME).exists():
    raise FileNotFoundError(
        f"Phase 2 checkpoint not found: {RESUME}\n"
        "Run the Phase 2 cell first and wait for it to complete."
    )

cmd = [
    sys.executable, "training/finance_pretrain.py",
    "--base_ckpt",    BASE_CKPT,
    "--ckpt_dir",     CKPT_DIR,
    "--phase",        "3",
    "--phase1_steps", "30000",
    "--phase2_steps", "8000",
    "--phase3_steps", "8000",
    "--phase4_steps", "3000",
    "--batch_size",   str(BATCH_SIZE),
    "--grad_accum_steps", str(GRAD_ACCUM_STEPS),
    "--seq_len",      str(SEQ_LEN),
    "--dtype",        "auto",
    "--log_every",    "100",
    "--mem_log_every", "100",
    "--cache_dir",    "/content/cache",
    "--save_every",   "1000",
    "--log_file",     f"{CKPT_DIR}/train.log",
    "--resume",       RESUME,
    "--auto_resume",
]
if USE_COMPILE:
    cmd.append("--compile")
if USE_GRAD_CHECKPOINT:
    cmd.append("--grad_checkpoint")
if LOOP_SCHEDULE != "off":
    cmd += ["--loop_schedule", LOOP_SCHEDULE]
if LOOP_SCHEDULE == "curriculum":
    cmd += ["--loop_tail_max", str(LOOP_TAIL_MAX), "--loop_tail_p", str(LOOP_TAIL_P), "--loop_seed", str(LOOP_SEED)]
if OPTIM8BIT:
    cmd.append("--optim8bit")
if REPLAY_RATIO > 0:
    cmd += ["--replay_ratio", str(REPLAY_RATIO)]

if ACT_CURRICULUM:
    cmd += [
        "--act_curriculum",
        "--act_anchor_step", str(ACT_ANCHOR_STEP),
        "--act_threshold_start", str(ACT_THRESHOLD_START),
        "--act_warmup_frac", str(ACT_WARMUP_FRAC),
        "--ponder_weight_start", str(PONDER_WEIGHT_START),
        "--ponder_weight_end", str(PONDER_WEIGHT_END),
    ]

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


In [ ]:
import subprocess, sys
from pathlib import Path

CKPT_DIR  = f"{REPO}/checkpoints/{CKPT_SUBDIR}"
BASE_CKPT = f"{REPO}/checkpoints/a100_v2_gpt2vocab/final.pt"
RESUME    = f"{CKPT_DIR}/phase3_final.pt"

if not Path(RESUME).exists():
    raise FileNotFoundError(
        f"Phase 3 checkpoint not found: {RESUME}\n"
        "Run the Phase 3 cell first and wait for it to complete."
    )

cmd = [
    sys.executable, "training/finance_pretrain.py",
    "--base_ckpt",    BASE_CKPT,
    "--ckpt_dir",     CKPT_DIR,
    "--phase",        "4",
    "--phase1_steps", "30000",
    "--phase2_steps", "8000",
    "--phase3_steps", "8000",
    "--phase4_steps", "3000",
    "--phase5_steps", "3000",
    "--batch_size",   str(BATCH_SIZE),
    "--grad_accum_steps", str(GRAD_ACCUM_STEPS),
    "--seq_len",      str(SEQ_LEN),
    "--dtype",        "auto",
    "--log_every",    "100",
    "--mem_log_every", "100",
    "--cache_dir",    "/content/cache",
    "--save_every",   "1000",
    "--log_file",     f"{CKPT_DIR}/train.log",
    "--resume",       RESUME,
    "--auto_resume",
]
if USE_COMPILE:
    cmd.append("--compile")
if USE_GRAD_CHECKPOINT:
    cmd.append("--grad_checkpoint")
if LOOP_SCHEDULE != "off":
    cmd += ["--loop_schedule", LOOP_SCHEDULE]
if LOOP_SCHEDULE == "curriculum":
    cmd += ["--loop_tail_max", str(LOOP_TAIL_MAX), "--loop_tail_p", str(LOOP_TAIL_P), "--loop_seed", str(LOOP_SEED)]
if OPTIM8BIT:
    cmd.append("--optim8bit")
if REPLAY_RATIO > 0:
    cmd += ["--replay_ratio", str(REPLAY_RATIO)]

if ACT_CURRICULUM:
    cmd += [
        "--act_curriculum",
        "--act_anchor_step", str(ACT_ANCHOR_STEP),
        "--act_threshold_start", str(ACT_THRESHOLD_START),
        "--act_warmup_frac", str(ACT_WARMUP_FRAC),
        "--ponder_weight_start", str(PONDER_WEIGHT_START),
        "--ponder_weight_end", str(PONDER_WEIGHT_END),
    ]

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


### Phase 5 — Risk-Management QA (Final Tuning)

Run this after Phase 4 completes. Training continues from `phase4_final.pt`.

- `FinGPT/fingpt-fiqa_qa` (~17K finance QA examples)

The 3,000-step final phase reinforces risk disclosure, uncertainty-aware wording, and a verification-first response tone.


In [ ]:
import subprocess, sys
from pathlib import Path

CKPT_DIR  = f"{REPO}/checkpoints/{CKPT_SUBDIR}"
BASE_CKPT = f"{REPO}/checkpoints/a100_v2_gpt2vocab/final.pt"
RESUME    = f"{CKPT_DIR}/phase4_final.pt"

if not Path(RESUME).exists():
    raise FileNotFoundError(
        f"Phase 4 checkpoint not found: {RESUME}\n"
        "Run the Phase 4 cell first and wait for it to complete."
    )

cmd = [
    sys.executable, "training/finance_pretrain.py",
    "--base_ckpt",    BASE_CKPT,
    "--ckpt_dir",     CKPT_DIR,
    "--phase",        "5",
    "--phase1_steps", "30000",
    "--phase2_steps", "8000",
    "--phase3_steps", "8000",
    "--phase4_steps", "3000",
    "--phase5_steps", "3000",
    "--batch_size",   str(BATCH_SIZE),
    "--grad_accum_steps", str(GRAD_ACCUM_STEPS),
    "--seq_len",      str(SEQ_LEN),
    "--dtype",        "auto",
    "--log_every",    "100",
    "--mem_log_every", "100",
    "--cache_dir",    "/content/cache",
    "--save_every",   "1000",
    "--log_file",     f"{CKPT_DIR}/train.log",
    "--resume",       RESUME,
    "--auto_resume",
]
if USE_COMPILE:
    cmd.append("--compile")
if USE_GRAD_CHECKPOINT:
    cmd.append("--grad_checkpoint")
if LOOP_SCHEDULE != "off":
    cmd += ["--loop_schedule", LOOP_SCHEDULE]
if LOOP_SCHEDULE == "curriculum":
    cmd += ["--loop_tail_max", str(LOOP_TAIL_MAX), "--loop_tail_p", str(LOOP_TAIL_P), "--loop_seed", str(LOOP_SEED)]
if OPTIM8BIT:
    cmd.append("--optim8bit")
if REPLAY_RATIO > 0:
    cmd += ["--replay_ratio", str(REPLAY_RATIO)]

if ACT_CURRICULUM:
    cmd += [
        "--act_curriculum",
        "--act_anchor_step", str(ACT_ANCHOR_STEP),
        "--act_threshold_start", str(ACT_THRESHOLD_START),
        "--act_warmup_frac", str(ACT_WARMUP_FRAC),
        "--ponder_weight_start", str(PONDER_WEIGHT_START),
        "--ponder_weight_end", str(PONDER_WEIGHT_END),
    ]

print("Command:", " ".join(cmd))
print("-" * 60)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExit code: {proc.returncode}")


## 7. Checkpoints


In [ ]:
from pathlib import Path

ckpt_dir = Path("checkpoints/finance_a100_v2")
ckpts = sorted(ckpt_dir.glob("*.pt"))

print(f"{'File':<30} {'Size':>8}")
print("-" * 40)
for p in ckpts:
    size_mb = p.stat().st_size / 1e6
    print(f"{p.name:<30} {size_mb:>7.1f} MB")

## 7b. Save Cache to Drive

Copy the NVMe tokenized cache back to Drive so the next session can skip tokenization.
**Run after all training phases complete, or before the session times out.**

In [ ]:
import shutil
from pathlib import Path

LOCAL_CACHE = "/content/cache"
DRIVE_CACHE = f"{REPO}/.cache"

local_path = Path(LOCAL_CACHE)
if local_path.exists():
    files = [f for f in local_path.rglob("*") if f.is_file()]
    if files:
        size_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f"Saving {len(files)} cache files ({size_mb:.0f} MB) from /content/cache → Drive ...")
        Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
        shutil.copytree(LOCAL_CACHE, DRIVE_CACHE, dirs_exist_ok=True)
        print(f"Done. Saved to {DRIVE_CACHE}")
    else:
        print("Local cache is empty, nothing to save.")
else:
    print("No local cache found.")


## QAT: INT8 量子化の頑健化(任意)

phase5 を INT8 dynamic 量子化すると finance PPL が大きく悪化する(kv_down 等が量子化ボトルネック)。
kv_down + 第2ボトルネック(shared_experts)+ dense FFN を狙った QAT で頑健化し、フル INT8 の
配布物を作る。手順: ①QAT 仕上げ → ②フル INT8 配布物生成 → ③評価。
詳細レポート: `training/report/kv_down_qat_findings.md`。

In [ ]:
# ① QAT 仕上げ(15層ターゲット + 整合正則化でループ保持。GPU 推奨)
!python3 training/qat_kv_down.py \
  --base_ckpt checkpoints/finance_a100_v2/phase5_final.pt \
  --targets recurrent.block.attn,shared_experts,prelude.0.ffn,coda.0.ffn \
  --consistency_lambda 1.0 --lr 1e-5 \
  --steps 1500 --n_loops 8 --batch_size 4 --seq_len 1024 --device cuda \
  --out checkpoints/finance_a100_v2/phase5_qat_floor.pt

In [ ]:
# ② フル INT8 配布物を生成(--keep_fp32 '' で全層 INT8)+ roundtrip 検証
!python3 training/make_mixed_int8.py \
  --ckpt checkpoints/finance_a100_v2/phase5_qat_floor.pt \
  --out  checkpoints/finance_a100_v2/phase5_qat_int8.pt \
  --keep_fp32 "" --verify

In [ ]:
# ③ 評価(INT8 dynamic は CPU 専用)
# D-view: base fp32 基準 / control: QAT 自身の fp32 基準(破壊チェック=fp32 が深さで改善するか)
!python3 training/eval_qat_compare.py \
  --base_ckpt checkpoints/finance_a100_v2/phase5_final.pt \
  --qat_ckpt  checkpoints/finance_a100_v2/phase5_qat_floor.pt \
  --eval_set finance --n_loops 1,2,4,8 --eval_max_chunks 30 --device cpu
!python3 training/eval_qat_compare.py \
  --base_ckpt checkpoints/finance_a100_v2/phase5_qat_floor.pt \
  --eval_set finance --n_loops 1,2,4,8 --eval_max_chunks 30 --device cpu

## 8. Inference Test


In [ ]:
import sys
sys.path.insert(0, REPO)

from pathlib import Path
import torch
from bushido_mythos import MythosConfig, BushidoMythos
from transformers import AutoTokenizer

device = torch.device("cuda")

ckpt_dir = Path(REPO) / "checkpoints/finance_a100_v2"
for name in ["phase5_final.pt", "phase4_final.pt", "phase3_final.pt", "final.pt", "phase2_final.pt", "phase1_final.pt"]:
    if (ckpt_dir / name).exists():
        ckpt_path = ckpt_dir / name
        break
else:
    ckpt_path = sorted(ckpt_dir.glob("step_*.pt"))[-1]

# Set to True only for checkpoints YOU created (pickle-based loading).
# Never enable for checkpoints from untrusted sources.
ALLOW_UNSAFE_CHECKPOINT = False

print(f"Loading: {ckpt_path}")
try:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
except Exception as _e:
    if not ALLOW_UNSAFE_CHECKPOINT:
        raise RuntimeError(
            f"weights_only=True failed: {_e}\n"
            "Set ALLOW_UNSAFE_CHECKPOINT = True in this cell only if this checkpoint\n"
            "was created by YOU and you trust it completely."
        ) from _e
    import warnings
    warnings.warn(f"weights_only=True failed, falling back (ALLOW_UNSAFE_CHECKPOINT=True): {_e}")
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
cfg  = MythosConfig(**ckpt["cfg"])
print(f"Checkpoint step={ckpt.get('step')} tag={ckpt.get('tag')} vocab={cfg.vocab_size:,}")
if ckpt.get("step", 0) < 52000:
    raise RuntimeError(
        f"This does not look like the completed Phase 5 checkpoint: step={ckpt.get('step')} tag={ckpt.get('tag')}. "
        "Check REPO/ckpt_dir and make sure phase5_final.pt exists."
    )
model = BushidoMythos(cfg).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()

from transformers import GPT2TokenizerFast

def load_verified_gpt2_tokenizer():
    attempts = [
        lambda: AutoTokenizer.from_pretrained("gpt2", use_fast=True, local_files_only=True),
        lambda: GPT2TokenizerFast.from_pretrained("gpt2", local_files_only=True),
        lambda: AutoTokenizer.from_pretrained("gpt2", use_fast=True),
        lambda: GPT2TokenizerFast.from_pretrained("gpt2"),
    ]
    errors = []
    for load in attempts:
        try:
            candidate = load()
            test_ids = candidate.encode("Hello", add_special_tokens=False)
            if test_ids:
                print(f"Tokenizer OK: {candidate.__class__.__name__}, test_ids={test_ids}")
                return candidate
            errors.append("loaded tokenizer returned empty ids for 'Hello'")
        except Exception as e:
            errors.append(f"{type(e).__name__}: {e}")
    raise RuntimeError(
        "GPT-2 tokenizer load failed or returned empty token ids. "
        "In Colab, run: !rm -rf ~/.cache/huggingface/hub/models--gpt2 and rerun this cell.\n"
        + "\n".join(errors[-3:])
    )

tok = load_verified_gpt2_tokenizer()

# Instruction-response format learned in Phases 3/4
_PREFIX   = "### Instruction:\n"
_RESPONSE = "\n\n### Response:\n"
_STOP     = "\n### "
_RISK_SUFFIX = (
    "\n\nPlease acknowledge uncertainty where relevant, include risk considerations, "
    "and note that outputs should be verified from authoritative sources before any trading action."
)

def chat(instruction: str, max_new_tokens: int = 200, temperature: float = 0.6,
         top_k: int = 40, n_loops: int = 8, repetition_penalty: float = 1.3):
    prompt = _PREFIX + instruction + _RISK_SUFFIX + _RESPONSE
    raw_ids = tok.encode(prompt, add_special_tokens=False)
    if not raw_ids:
        raw_ids = tok(prompt, add_special_tokens=False).get("input_ids", [])
    if not raw_ids:
        raise RuntimeError(
            "Tokenizer returned empty ids even after verified load. "
            "Clear the Colab HuggingFace cache with: !rm -rf ~/.cache/huggingface/hub/models--gpt2"
        )
    # Match training/finance_pretrain.py: GPT-2 token ids are clamped into the model vocab.
    ids = [min(int(t), cfg.vocab_size - 1) for t in raw_ids]
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model.generate(input_ids, max_new_tokens=max_new_tokens,
                             temperature=temperature, top_k=top_k, n_loops=n_loops,
                             repetition_penalty=repetition_penalty)
    result = tok.decode(out[0, len(ids):].tolist(), skip_special_tokens=True)
    stop_idx = result.find(_STOP)
    return result[:stop_idx].strip() if stop_idx != -1 else result.strip()

prompts = [
    "Explain the risks of trading with high leverage.",
    "What is the Federal Reserve's role in controlling inflation?",
    "How should a trader approach risk management when holding a volatile position overnight?",
]

for p in prompts:
    print(f"\n{'='*60}")
    print(f"[Instruction] {p}")
    print(f"[Response]    {chat(p)}")


## Tips

**If you hit OOM:** halve `BATCH_SIZE` and rerun from the GPU-check cell.

**To reduce session-disconnect loss:** set `--save_every 1000` to save more frequently. You can resume with `--auto_resume`.

**If you want to use `--compile` on T4:** explicitly set `--dtype float16`; bfloat16 compile is not supported on T4.

**Cache:** if `.cache/wikitext103_gpt2_50257_v1.pt` exists on Drive, tokenization is skipped.
